# Distillation Offset-Free MPC Observer Pole Sweep (Temporary)

This notebook runs a temporary observer-pole sweep for the distillation offset-free MPC baseline.

Workflow:
1. Use the distillation baseline notebook defaults, but force `run_mode = "nominal"` and `n_tests = 2`.
2. Open Aspen once to load the steady-state context and scaling artifacts.
3. For each candidate pole set, open a fresh Aspen session, build `L`, run the baseline MPC, save the bundle, optionally save per-run plots, and close Aspen.
4. Save a sweep summary table and comparison figures at the end.

This notebook is intentionally notebook-local and temporary. It does not change shared distillation defaults.


In [ ]:
from datetime import datetime
from pathlib import Path
import json
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from systems.distillation import get_distillation_notebook_defaults
from utils.notebook_setup import prepare_distillation_notebook_env, print_grouped_notebook_summary

NB = get_distillation_notebook_defaults("baseline")
RUN_MODE = "nominal"
DISTURBANCE_PROFILE = "none"
STYLE_PROFILE = NB["style_profile"]
SAVE_PDF = NB["save_pdf"]
ASPEN_PRESET = NB["aspen_preset"]
ASPEN_PATH_OVERRIDE = NB["aspen_path_override"]
SNAPS_PATH_OVERRIDE = NB["snaps_path_override"]
ASPEN_ROOT_OVERRIDE = NB["aspen_root_override"]
DISTILLATION_VISIBLE = NB["distillation_visible"]
DISTILLATION_DATA_DIR_OVERRIDE = NB["data_dir_override"]
DISTILLATION_RESULTS_DIR_OVERRIDE = NB["results_dir_override"]
N_TESTS_OVERRIDE = 2
SET_POINTS_LEN_OVERRIDE = NB["set_points_len_override"]
TEST_CYCLE_OVERRIDE = NB["test_cycle_override"]
PLOT_START_EPISODE_OVERRIDE = 1
SAVE_PER_RUN_PLOTS = True

REPO_ROOT, DATA_DIR, RESULT_DIR, DISTURBANCE_PROFILE, DYN_PATH, SNAPS_PATH, ASPEN_SOURCE = prepare_distillation_notebook_env(
    run_mode=RUN_MODE,
    disturbance_profile=DISTURBANCE_PROFILE,
    family="baseline",
    aspen_preset=ASPEN_PRESET,
    dyn_path_override=ASPEN_PATH_OVERRIDE,
    snaps_path_override=SNAPS_PATH_OVERRIDE,
    aspen_root_override=ASPEN_ROOT_OVERRIDE,
    data_dir_override=DISTILLATION_DATA_DIR_OVERRIDE,
    results_dir_override=DISTILLATION_RESULTS_DIR_OVERRIDE,
)
os.chdir(REPO_ROOT)

SESSION_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
SWEEP_NAME = f"distillation_observer_pole_sweep_nominal_{SESSION_STAMP}"
SWEEP_DATA_DIR = DATA_DIR / "observer_pole_sweep_temp" / SESSION_STAMP
SWEEP_RESULT_DIR = RESULT_DIR / "observer_pole_sweep_temp" / SESSION_STAMP
SWEEP_DATA_DIR.mkdir(parents=True, exist_ok=True)
SWEEP_RESULT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
from Simulation.mpc import MpcSolverGeneral
from systems.distillation.data_io import load_distillation_system_data
from systems.distillation.labels import DISTILLATION_SYSTEM_METADATA
from systems.distillation.plant import build_distillation_system, distillation_system_stepper
from systems.distillation.scenarios import build_distillation_disturbance_schedule
from utils.helpers import apply_min_max, reverse_min_max
from utils.mpc_baseline_runner import run_offsetfree_mpc
from utils.observer import compute_observer_gain
from utils.plotting import plot_baseline_mpc_results
from utils.rewards import make_reward_fn_relative_QR


In [ ]:
# Load the baseline system setup and open Aspen once to get the steady-state/scaling context.
SYS = NB["system_setup"]
nominal_conditions = SYS["nominal_conditions"].copy()
ss_inputs = SYS["ss_inputs"].copy()
u_min = SYS["input_bounds"]["u_min"].copy()
u_max = SYS["input_bounds"]["u_max"].copy()
setpoint_y = SYS["setpoint_range_phys"].copy()

reference_system = build_distillation_system(
    path=DYN_PATH,
    ss_inputs=ss_inputs,
    initialization_point=nominal_conditions,
    delta_t=SYS["delta_t_hours"],
    visible=DISTILLATION_VISIBLE,
)
try:
    steady_states = {
        "ss_inputs": np.asarray(reference_system.ss_inputs, float).copy(),
        "y_ss": np.asarray(reference_system.y_ss, float).copy(),
    }
finally:
    try:
        reference_system.close(SNAPS_PATH)
    except Exception as exc:
        print(f"Could not close reference Aspen session cleanly: {exc}")

system_data = load_distillation_system_data(
    REPO_ROOT,
    steady_states,
    setpoint_y,
    u_min,
    u_max,
    data_override=DISTILLATION_DATA_DIR_OVERRIDE,
)
A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]
inputs_number = int(B_aug.shape[1])
y_sp_scenario_phys = SYS["rl_setpoints_phys"].copy()
y_ss_scaled = apply_min_max(steady_states["y_ss"], data_min[inputs_number:], data_max[inputs_number:])
y_sp_scenario = apply_min_max(y_sp_scenario_phys, data_min[inputs_number:], data_max[inputs_number:]) - y_ss_scaled


In [ ]:
# Controller, reward, and temporary sweep configuration.
CTRL = NB["controller"]
REWARD_CFG = NB["reward"]
RUN_PROFILE = NB["run_profiles"][(RUN_MODE, DISTURBANCE_PROFILE)]

n_inputs = int(B_aug.shape[1])
predict_h = CTRL["predict_h"]
cont_h = CTRL["cont_h"]
Q1_penalty = CTRL["Q1_penalty"]
Q2_penalty = CTRL["Q2_penalty"]
R1_penalty = CTRL["R1_penalty"]
R2_penalty = CTRL["R2_penalty"]
USE_SHIFTED_MPC_WARM_START = CTRL["use_shifted_mpc_warm_start"]

n_tests = int(RUN_PROFILE["n_tests"] if N_TESTS_OVERRIDE is None else N_TESTS_OVERRIDE)
set_points_len = int(RUN_PROFILE["set_points_len"] if SET_POINTS_LEN_OVERRIDE is None else SET_POINTS_LEN_OVERRIDE)
warm_start = int(RUN_PROFILE["warm_start"])
if warm_start >= n_tests:
    warm_start = max(0, n_tests - 1)
TEST_CYCLE = list(RUN_PROFILE["test_cycle"] if TEST_CYCLE_OVERRIDE is None else TEST_CYCLE_OVERRIDE)
PLOT_START_EPISODE = int(RUN_PROFILE.get("plot_start_episode", 1) if PLOT_START_EPISODE_OVERRIDE is None else PLOT_START_EPISODE_OVERRIDE)

reward_params, reward_fn = make_reward_fn_relative_QR(data_min, data_max, n_inputs, **REWARD_CFG)

DISTILLATION_OBSERVER_POLE_CANDIDATES = {
    "p00_old_aggressive_reference": np.array([0.0115, 0.0320, 0.0350, 0.0410, 0.0419, 0.0748, 0.4104]),
    "p01_mid_current_test": np.array([0.35, 0.40, 0.45, 0.50, 0.55, 0.80, 0.90]),
    "p02_fast_soft_offset": np.array([0.10, 0.13, 0.16, 0.20, 0.25, 0.65, 0.85]),
    "p03_fast_slow_offset": np.array([0.10, 0.15, 0.20, 0.25, 0.30, 0.75, 0.90]),
    "p04_fast_polymer_like_offset": np.array([0.12, 0.18, 0.24, 0.30, 0.36, 0.80, 0.93]),
    "p05_fast_with_one_very_slow_offset": np.array([0.12, 0.16, 0.22, 0.28, 0.35, 0.70, 0.96]),
    "p06_moderate_fast_a": np.array([0.20, 0.25, 0.30, 0.35, 0.40, 0.75, 0.88]),
    "p07_moderate_fast_b": np.array([0.22, 0.28, 0.34, 0.40, 0.46, 0.78, 0.90]),
    "p08_moderate_fast_c": np.array([0.25, 0.30, 0.35, 0.40, 0.45, 0.80, 0.92]),
    "p09_moderate_fast_d": np.array([0.28, 0.33, 0.38, 0.43, 0.48, 0.82, 0.93]),
    "p10_current_slightly_faster": np.array([0.28, 0.34, 0.40, 0.46, 0.52, 0.78, 0.88]),
    "p11_current_balanced": np.array([0.32, 0.38, 0.44, 0.50, 0.56, 0.80, 0.90]),
    "p12_current_slightly_slower": np.array([0.38, 0.43, 0.48, 0.53, 0.58, 0.82, 0.92]),
    "p13_current_more_offset_slow": np.array([0.35, 0.40, 0.45, 0.50, 0.55, 0.86, 0.94]),
    "p14_current_less_offset_slow": np.array([0.35, 0.40, 0.45, 0.50, 0.55, 0.72, 0.86]),
    "p15_slow_physical_a": np.array([0.45, 0.50, 0.55, 0.60, 0.65, 0.85, 0.93]),
    "p16_slow_physical_b": np.array([0.50, 0.55, 0.60, 0.65, 0.70, 0.88, 0.94]),
    "p17_slow_physical_c": np.array([0.55, 0.60, 0.65, 0.70, 0.75, 0.90, 0.95]),
    "p18_slow_physical_d": np.array([0.60, 0.64, 0.68, 0.72, 0.76, 0.90, 0.96]),
    "p19_uniform_fast": np.array([0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]),
    "p20_uniform_mid": np.array([0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]),
    "p21_uniform_slow": np.array([0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85]),
    "p22_polymer_style_fast": np.array([0.30, 0.34, 0.38, 0.42, 0.46, 0.91, 0.97]),
    "p23_polymer_style_mid": np.array([0.35, 0.39, 0.43, 0.47, 0.51, 0.90, 0.96]),
    "p24_polymer_style_slow": np.array([0.42, 0.46, 0.50, 0.54, 0.58, 0.91, 0.97]),
    "p25_one_slow_offset_fast": np.array([0.18, 0.24, 0.30, 0.36, 0.42, 0.55, 0.92]),
    "p26_one_slow_offset_mid": np.array([0.30, 0.36, 0.42, 0.48, 0.54, 0.65, 0.92]),
    "p27_two_slow_offsets_wide_gap": np.array([0.25, 0.32, 0.39, 0.46, 0.53, 0.82, 0.97]),
    "p28_conservative_mixed": np.array([0.40, 0.46, 0.52, 0.58, 0.64, 0.86, 0.95]),
    "p29_very_conservative_mixed": np.array([0.50, 0.56, 0.62, 0.68, 0.74, 0.90, 0.97]),
}
ACTIVE_CANDIDATE_KEYS = list(DISTILLATION_OBSERVER_POLE_CANDIDATES.keys())


In [ ]:
print_grouped_notebook_summary(
    "Distillation observer pole sweep summary",
    {
        "Paths": {
            "Repo root": REPO_ROOT,
            "Data dir": DATA_DIR,
            "Results dir": RESULT_DIR,
            "Aspen source": ASPEN_SOURCE,
            "Dyn path": DYN_PATH,
            "Snaps path": SNAPS_PATH,
            "Sweep data dir": SWEEP_DATA_DIR,
            "Sweep result dir": SWEEP_RESULT_DIR,
        },
        "Run setup": {
            "Run mode": RUN_MODE,
            "Disturbance profile": DISTURBANCE_PROFILE,
            "n_tests": n_tests,
            "set_points_len": set_points_len,
            "warm_start": warm_start,
            "test_cycle": TEST_CYCLE,
            "use_shifted_mpc_warm_start": USE_SHIFTED_MPC_WARM_START,
            "save_per_run_plots": SAVE_PER_RUN_PLOTS,
        },
        "System / controller": {
            "delta_t_hours": SYS["delta_t_hours"],
            "predict_h": predict_h,
            "cont_h": cont_h,
            "setpoints_phys": y_sp_scenario_phys.tolist(),
        },
        "Sweep": {
            "n_candidates": len(ACTIVE_CANDIDATE_KEYS),
            "first_candidate": ACTIVE_CANDIDATE_KEYS[0] if ACTIVE_CANDIDATE_KEYS else None,
            "last_candidate": ACTIVE_CANDIDATE_KEYS[-1] if ACTIVE_CANDIDATE_KEYS else None,
        },
        "Reward": reward_params,
    },
)


In [ ]:
def build_run_components(poles):
    poles = np.asarray(poles, dtype=float)
    system = build_distillation_system(
        path=DYN_PATH,
        ss_inputs=ss_inputs,
        initialization_point=nominal_conditions,
        delta_t=SYS["delta_t_hours"],
        visible=DISTILLATION_VISIBLE,
    )
    disturbance_schedule = None
    if RUN_MODE == "disturb":
        nominal_feed = float(system.feed.FmR.Value)
        disturbance_schedule = build_distillation_disturbance_schedule(
            RUN_MODE,
            DISTURBANCE_PROFILE,
            total_steps=n_tests * set_points_len * len(y_sp_scenario_phys),
            nominal_feed=nominal_feed,
        )

    L = compute_observer_gain(A_aug, C_aug, poles)
    mpc_obj = MpcSolverGeneral(
        A_aug,
        B_aug,
        C_aug,
        Q_out=np.array([Q1_penalty, Q2_penalty], float),
        R_in=np.array([R1_penalty, R2_penalty], float),
        NP=predict_h,
        NC=cont_h,
    )
    mpc_cfg = {
        "run_mode": RUN_MODE,
        "n_tests": n_tests,
        "set_points_len": set_points_len,
        "warm_start": warm_start,
        "test_cycle": TEST_CYCLE,
        "predict_h": predict_h,
        "cont_h": cont_h,
        "use_shifted_mpc_warm_start": USE_SHIFTED_MPC_WARM_START,
        "Q1_penalty": Q1_penalty,
        "Q2_penalty": Q2_penalty,
        "R1_penalty": R1_penalty,
        "R2_penalty": R2_penalty,
        "nominal_qi": CTRL["nominal_qi"],
        "nominal_qs": CTRL["nominal_qs"],
        "nominal_ha": CTRL["nominal_ha"],
        "qi_change": CTRL["qi_change"],
        "qs_change": CTRL["qs_change"],
        "ha_change": CTRL["ha_change"],
        "b_min": system_data["b_min"],
        "b_max": system_data["b_max"],
    }
    runtime_ctx = {
        "system": system,
        "MPC_obj": mpc_obj,
        "steady_states": steady_states,
        "data_min": data_min,
        "data_max": data_max,
        "A_aug": A_aug,
        "B_aug": B_aug,
        "C_aug": C_aug,
        "y_sp_scenario": y_sp_scenario,
        "reward_fn": reward_fn,
        "system_metadata": DISTILLATION_SYSTEM_METADATA,
        "poles": poles,
        "L": L,
        "disturbance_schedule": disturbance_schedule,
        "system_stepper": distillation_system_stepper,
        "disturbance_labels": DISTILLATION_SYSTEM_METADATA["disturbance_labels"],
        "u_lower_phys": u_min,
        "u_upper_phys": u_max,
    }
    return system, mpc_cfg, runtime_ctx, L


def summarize_bundle(label, poles, L, bundle, bundle_path, plot_dir):
    time_in_sub = int(bundle["time_in_sub_episodes"])
    last_slice = slice(max(0, bundle["nFE"] - time_in_sub), bundle["nFE"])
    sat = bundle.get("input_saturation_summary", {})
    return {
        "label": label,
        "status": "ok",
        "reward_step_mean": float(np.mean(bundle["rewards_step"])),
        "reward_step_std": float(np.std(bundle["rewards_step"])),
        "reward_last_episode_mean": float(np.mean(bundle["rewards_step"][last_slice])),
        "reward_last_episode_std": float(np.std(bundle["rewards_step"][last_slice])),
        "reward_min_step": float(np.min(bundle["rewards_step"])),
        "output1_mae_mean": float(np.mean(np.abs(bundle["delta_y_storage"][:, 0]))),
        "output2_mae_mean": float(np.mean(np.abs(bundle["delta_y_storage"][:, 1]))),
        "output1_mae_last_episode": float(np.mean(np.abs(bundle["delta_y_storage"][last_slice, 0]))),
        "output2_mae_last_episode": float(np.mean(np.abs(bundle["delta_y_storage"][last_slice, 1]))),
        "u1_lower_sat_frac": float(np.asarray(sat.get("lower_fraction", [np.nan, np.nan]), float)[0]),
        "u1_upper_sat_frac": float(np.asarray(sat.get("upper_fraction", [np.nan, np.nan]), float)[0]),
        "u2_lower_sat_frac": float(np.asarray(sat.get("lower_fraction", [np.nan, np.nan]), float)[1]),
        "u2_upper_sat_frac": float(np.asarray(sat.get("upper_fraction", [np.nan, np.nan]), float)[1]),
        "gain_fro_norm": float(np.linalg.norm(L)),
        "bundle_path": str(bundle_path),
        "plot_dir": str(plot_dir) if plot_dir is not None else None,
        "poles": [float(x) for x in poles],
    }


def run_one_candidate(label, poles):
    system, mpc_cfg, runtime_ctx, L = build_run_components(poles)
    bundle_path = SWEEP_DATA_DIR / f"{label}.pickle"
    per_run_plot_dir = None
    try:
        bundle = run_offsetfree_mpc(mpc_cfg=mpc_cfg, runtime_ctx=runtime_ctx)
        bundle["observer_label"] = label
        bundle["observer_poles"] = np.asarray(poles, dtype=float).copy()
        bundle["observer_gain"] = np.asarray(L, dtype=float).copy()
        bundle["sweep_name"] = SWEEP_NAME
        with open(bundle_path, "wb") as handle:
            pickle.dump(bundle, handle)

        if SAVE_PER_RUN_PLOTS:
            per_run_plot_dir = plot_baseline_mpc_results(
                result_bundle=bundle,
                plot_cfg={
                    "directory": SWEEP_RESULT_DIR,
                    "prefix_name": f"{label}_{RUN_MODE}_{DISTURBANCE_PROFILE}",
                    "start_episode": PLOT_START_EPISODE,
                    "save_pdf": SAVE_PDF,
                    "style_profile": STYLE_PROFILE,
                },
            )

        summary_row = summarize_bundle(label, poles, L, bundle, bundle_path, per_run_plot_dir)
        return summary_row, bundle
    finally:
        try:
            system.close(SNAPS_PATH)
        except Exception as exc:
            print(f"[{label}] Could not close Aspen cleanly: {exc}")


In [ ]:
summary_rows = []
run_bundles = {}
run_errors = {}

for idx, label in enumerate(ACTIVE_CANDIDATE_KEYS, start=1):
    poles = DISTILLATION_OBSERVER_POLE_CANDIDATES[label]
    print(f"[{idx:02d}/{len(ACTIVE_CANDIDATE_KEYS):02d}] Starting {label}: {np.asarray(poles, float)}")
    try:
        summary_row, bundle = run_one_candidate(label, poles)
        summary_rows.append(summary_row)
        run_bundles[label] = bundle
        print(
            f"[{label}] done | reward_step_mean={summary_row['reward_step_mean']:.4f} | "
            f"reward_last_episode_mean={summary_row['reward_last_episode_mean']:.4f}"
        )
    except Exception as exc:
        error_text = repr(exc)
        run_errors[label] = error_text
        summary_rows.append({
            "label": label,
            "status": "failed",
            "error": error_text,
            "poles": [float(x) for x in np.asarray(poles, float)],
        })
        print(f"[{label}] failed: {error_text}")

summary_df = pd.DataFrame(summary_rows)
success_df = summary_df[summary_df["status"] == "ok"].copy() if "status" in summary_df else pd.DataFrame()
summary_df


In [ ]:
SUMMARY_CSV_PATH = SWEEP_RESULT_DIR / "observer_pole_sweep_summary.csv"
MANIFEST_PATH = SWEEP_RESULT_DIR / "observer_pole_sweep_manifest.json"
REWARD_PLOT_PATH = SWEEP_RESULT_DIR / "observer_pole_sweep_reward_summary.png"
MAE_PLOT_PATH = SWEEP_RESULT_DIR / "observer_pole_sweep_tracking_mae_summary.png"
SUBEPISODE_PLOT_PATH = SWEEP_RESULT_DIR / "observer_pole_sweep_avg_reward_traces.png"

summary_df.to_csv(SUMMARY_CSV_PATH, index=False)
manifest = {
    "sweep_name": SWEEP_NAME,
    "run_mode": RUN_MODE,
    "disturbance_profile": DISTURBANCE_PROFILE,
    "n_candidates": len(ACTIVE_CANDIDATE_KEYS),
    "n_success": int(len(success_df)),
    "n_failed": int(len(run_errors)),
    "summary_csv": str(SUMMARY_CSV_PATH),
    "bundle_paths": {label: row["bundle_path"] for _, row in success_df[["label", "bundle_path"]].iterrows()} if not success_df.empty else {},
    "failed": run_errors,
}
with open(MANIFEST_PATH, "w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

if not success_df.empty:
    display(success_df.sort_values("reward_last_episode_mean", ascending=False).reset_index(drop=True))

    plot_df = success_df.sort_values("reward_last_episode_mean", ascending=True).reset_index(drop=True)
    labels = plot_df["label"].tolist()
    fig_height = max(8.0, 0.32 * len(plot_df))

    fig, axes = plt.subplots(1, 2, figsize=(16, fig_height), constrained_layout=True)
    axes[0].barh(labels, plot_df["reward_step_mean"], color="tab:blue")
    axes[0].set_title("Mean step reward")
    axes[0].set_xlabel("Reward")
    axes[1].barh(labels, plot_df["reward_last_episode_mean"], color="tab:green")
    axes[1].set_title("Last-episode mean reward")
    axes[1].set_xlabel("Reward")
    fig.savefig(REWARD_PLOT_PATH, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    fig, axes = plt.subplots(1, 2, figsize=(16, fig_height), constrained_layout=True)
    axes[0].barh(labels, plot_df["output1_mae_last_episode"], color="tab:orange")
    axes[0].set_title(f"Last-episode scaled MAE: {DISTILLATION_SYSTEM_METADATA['output_labels'][0]}")
    axes[0].set_xlabel("Scaled MAE")
    axes[1].barh(labels, plot_df["output2_mae_last_episode"], color="tab:red")
    axes[1].set_title(f"Last-episode scaled MAE: {DISTILLATION_SYSTEM_METADATA['output_labels'][1]}")
    axes[1].set_xlabel("Scaled MAE")
    fig.savefig(MAE_PLOT_PATH, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(12, 7), constrained_layout=True)
    for label, bundle in run_bundles.items():
        avg_rewards = np.asarray(bundle.get("avg_rewards", []), float)
        if avg_rewards.size == 0:
            continue
        ax.plot(np.arange(1, avg_rewards.size + 1), avg_rewards, marker="o", linewidth=1.5, label=label)
    ax.set_title("Average reward by subepisode")
    ax.set_xlabel("Subepisode")
    ax.set_ylabel("Average reward")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8)
    fig.savefig(SUBEPISODE_PLOT_PATH, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)

print(f"Summary CSV: {SUMMARY_CSV_PATH}")
print(f"Manifest JSON: {MANIFEST_PATH}")
print(f"Reward summary plot: {REWARD_PLOT_PATH}")
print(f"Tracking MAE plot: {MAE_PLOT_PATH}")
print(f"Avg-reward trace plot: {SUBEPISODE_PLOT_PATH}")
if run_errors:
    print("Failed candidates:")
    for label, error_text in run_errors.items():
        print(f"  {label}: {error_text}")
